In [1]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [6]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

df = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train: {df.shape}  |  Test: {test.shape}')
print(f'Missing values in train:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print(f'Missing values in test:\n{test.isnull().sum()[test.isnull().sum() > 0]}')

Train: (699635, 23)  |  Test: (299844, 22)
Missing values in train:
Arrival Delay in Minutes    292
dtype: int64
Missing values in test:
Arrival Delay in Minutes    130
dtype: int64


In [7]:
# Keep ID separate
ids = df["id"].copy() if "id" in df.columns else None

# Remove ID from features if it is not a meaningful predictor
X = df.drop(columns=["id", "satisfaction"], errors="ignore")
y = df["satisfaction"]

numerical_cols = [
    "Age", "Flight Distance",
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location", "Food and drink",
    "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service",
    "Leg room service", "Baggage handling",
    "Checkin service", "Cleanliness",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

categorical_cols = [
    "Gender", "Type of Travel", "Class", "Customer Type"
]

# Encode target labels
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [9]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [10]:
catboost_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", CatBoostClassifier(
        iterations=1500,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=19,
        verbose=100,
        allow_writing_files=False
    ))
])

# Train the model
catboost_model.fit(X_train, y_train)

print("CatBoost training completed!")

0:	total: 175ms	remaining: 4m 22s
100:	total: 4.12s	remaining: 57.1s
200:	total: 8.24s	remaining: 53.3s
300:	total: 12.2s	remaining: 48.4s
400:	total: 16.1s	remaining: 44.1s
500:	total: 20.2s	remaining: 40.2s
600:	total: 24.4s	remaining: 36.5s
700:	total: 28.4s	remaining: 32.4s
800:	total: 31.8s	remaining: 27.8s
900:	total: 35s	remaining: 23.3s
1000:	total: 38.3s	remaining: 19.1s
1100:	total: 41.5s	remaining: 15s
1200:	total: 44.8s	remaining: 11.1s
1300:	total: 48.2s	remaining: 7.37s
1400:	total: 51.4s	remaining: 3.63s
1499:	total: 54.6s	remaining: 0us
CatBoost training completed!


In [11]:
y_pred = catboost_model.predict(X_test).ravel().astype(int)
y_prob = catboost_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(target_encoder.classes_)),
    target_names=[str(c) for c in target_encoder.classes_],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.9248250873669842

Classification Report:
              precision    recall  f1-score   support

       False       0.92      0.94      0.93     77859
        True       0.93      0.90      0.91     62068

    accuracy                           0.92    139927
   macro avg       0.92      0.92      0.92    139927
weighted avg       0.92      0.92      0.92    139927

Confusion Matrix:
[[73386  4473]
 [ 6046 56022]]
ROC-AUC: 0.9575816549881632


In [13]:
# Load new data without the target column
new_data = pd.read_csv("test.csv")

# Preserve IDs
ids = new_data["id"].copy()

# Remove ID and target from model input
X_new = new_data.drop(
    columns=["id", "satisfaction"],
    errors="ignore"
)

# Generate predictions
new_predictions = (
    catboost_model.predict(X_new)
    .ravel()
    .astype(int)
)

predicted_labels = target_encoder.inverse_transform(
    new_predictions
)

# Prediction probabilities
probabilities = catboost_model.predict_proba(X_new)

# Create output
results = pd.DataFrame({
    "id": ids,
    "Predicted_Satisfaction": predicted_labels,
    "Probability_Class_0": probabilities[:, 0],
    "Probability_Class_1": probabilities[:, 1]
})

print(results.head())

       id  Predicted_Satisfaction  Probability_Class_0  Probability_Class_1
0  699635                   False             0.737370             0.262630
1  699636                    True             0.114530             0.885470
2  699637                    True             0.115946             0.884054
3  699638                   False             0.936820             0.063180
4  699639                   False             0.946269             0.053731


In [15]:
results[['id','Predicted_Satisfaction']].to_csv("catboost_predictions.csv", index=False)
